# Laboratorio 7 - Spark MLlib
CC3066 - Data Science  

## Análisis exploratorio avanzado y segmentación

### Integrantes

- María José Girón Isidro
- Leonardo Dufrey Mejía Mejía
- Mia Alejandra Fuentes Mérida

---

## Objetivo

En este notebook se realiza la preparación, análisis exploratorio y
segmentación de los datos de Personas de la Encuesta Nacional de Empleo
e Ingresos Continua (ENEIC).

Para el desarrollo se utilizan los cuatro trimestres correspondientes
al año 2025. El primer trimestre de 2026 se mantiene separado para la
evaluación final.

In [25]:
import os
import math
import pandas as pd
import numpy as np
import openpyxl
import gc

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.stat import Correlation
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

import matplotlib.pyplot as plt

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Laboratorio7-ENEIC")
    .getOrCreate()
)

print("Versión de Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 22:08:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Versión de Spark: 3.5.1


In [4]:
print("Estoy trabajando en:")
print(os.getcwd())
print("Archivos disponibles:")
print(os.listdir("."))
print(os.listdir("./data"))

Estoy trabajando en:
/opt/app/CC3084-Laboratorio7
Archivos disponibles:
['data', 'Laboratorio7.ipynb', '.ipynb_checkpoints']
['Personas_ENEIC_T1_2025.xlsx', 'diccionarios', '.ipynb_checkpoints', 'Personas-ENEIC-T2-2025.xlsx', 'Base-de-datos-Personas-ENEIC-IV-2025.xlsx', 'Base-de-datos-Personas-ENEIC-I-2026.xlsx', 'Base-de-datos-Personas-ENEIC-III-2025.xlsx']


In [5]:
DATA_DIR = "./data"

ARCHIVOS = {
    "2025T1": {
        "ruta": f"{DATA_DIR}/Personas_ENEIC_T1_2025.xlsx",
        "anio": 2025,
        "trimestre": 1
    },
    "2025T2": {
        "ruta": f"{DATA_DIR}/Personas-ENEIC-T2-2025.xlsx",
        "anio": 2025,
        "trimestre": 2
    },
    "2025T3": {
        "ruta": f"{DATA_DIR}/Base-de-datos-Personas-ENEIC-III-2025.xlsx",
        "anio": 2025,
        "trimestre": 3
    },
    "2025T4": {
        "ruta": f"{DATA_DIR}/Base-de-datos-Personas-ENEIC-IV-2025.xlsx",
        "anio": 2025,
        "trimestre": 4
    },
    "2026T1": {
        "ruta": f"{DATA_DIR}/Base-de-datos-Personas-ENEIC-I-2026.xlsx",
        "anio": 2026,
        "trimestre": 1
    }
}

for periodo, info in ARCHIVOS.items():
    print(periodo, "->", os.path.exists(info["ruta"]))

2025T1 -> True
2025T2 -> True
2025T3 -> True
2025T4 -> True
2026T1 -> True


In [6]:
COLUMNAS_REQUERIDAS = [
    "P05D01",       # salario mensual
    "P02A03",       # edad
    "P05C07A",      # antigüedad años
    "P05C07B",      # antigüedad meses
    "P05H01A",      # horas semanales
    "P03A03A",      # nivel educativo
    "P05C16",       # categoría ocupacional
    "DOMINIO",      # dominio
    "OCUPADOS",     # condición de ocupado
    "NUM_HOGAR",
    "NUM_PERSONA",
    "FACTOR",
    "ANIO",
    "TRIMESTRE"
]

RENOMBRES = {
    "P05D01": "salario_mensual",
    "P02A03": "edad",
    "P05C07A": "antiguedad_anios",
    "P05C07B": "antiguedad_meses",
    "P05H01A": "horas_semanales",
    "P03A03A": "nivel_educativo",
    "P05C16": "categoria_ocupacional",
    "DOMINIO": "dominio",
    "OCUPADOS": "ocupado"
}

In [9]:
for periodo, info in ARCHIVOS.items():

    encabezado = pd.read_excel(
        info["ruta"],
        engine="openpyxl",
        nrows=0
    )

    columnas = list(encabezado.columns)

    faltantes = [
        col for col in COLUMNAS_REQUERIDAS
        if col not in columnas
    ]

    print(f"\n{periodo}")
    print(f"Columnas totales: {len(columnas)}")
    print(f"Columnas requeridas faltantes: {faltantes}")


2025T1
Columnas totales: 270
Columnas requeridas faltantes: []

2025T2
Columnas totales: 270
Columnas requeridas faltantes: []

2025T3
Columnas totales: 270
Columnas requeridas faltantes: []

2025T4
Columnas totales: 302
Columnas requeridas faltantes: []

2026T1
Columnas totales: 270
Columnas requeridas faltantes: []


In [10]:
PARQUET_DIR = "./data/parquet"

os.makedirs(PARQUET_DIR, exist_ok=True)

print("Directorio creado:", PARQUET_DIR)

Directorio creado: ./data/parquet


In [14]:
ruta_t1 = ARCHIVOS["2025T1"]["ruta"]

pdf_t1 = pd.read_excel(
    ruta_t1,
    engine="openpyxl",
    usecols=COLUMNAS_REQUERIDAS
)

print("Dimensiones:", pdf_t1.shape)
display(pdf_t1.head())

Dimensiones: (51588, 14)


,ANIO,TRIMESTRE,DOMINIO,NUM_HOGAR,FACTOR,NUM_PERSONA,P02A03,P03A03A,P05C07A,P05C07B,P05C16,P05D01,P05H01A,OCUPADOS
0,2025,2,2,13796,338,3,14,3.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,2,2,13796,338,1,42,5.0,23.0,0.0,6.0,NaN,35.0,1.0
2,2025,2,2,13796,338,2,42,5.0,20.0,0.0,1.0,7000.0,1.0,1.0
3,2025,2,2,13796,338,5,9,2.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,2,2,13796,338,4,11,2.0,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
pdf_t1.info()


<class 'pandas.DataFrame'>
RangeIndex: 51588 entries, 0 to 51587
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ANIO         51588 non-null  int64  
 1   TRIMESTRE    51588 non-null  int64  
 2   DOMINIO      51588 non-null  int64  
 3   NUM_HOGAR    51588 non-null  int64  
 4   FACTOR       51588 non-null  int64  
 5   NUM_PERSONA  51588 non-null  int64  
 6   P02A03       51588 non-null  int64  
 7   P03A03A      44660 non-null  float64
 8   P05C07A      22273 non-null  float64
 9   P05C07B      22273 non-null  float64
 10  P05C16       22273 non-null  float64
 11  P05D01       13419 non-null  float64
 12  P05H01A      22273 non-null  float64
 13  OCUPADOS     22273 non-null  float64
dtypes: float64(7), int64(7)
memory usage: 5.5 MB


In [16]:
display(pdf_t1.head(10))

,ANIO,TRIMESTRE,DOMINIO,NUM_HOGAR,FACTOR,NUM_PERSONA,P02A03,P03A03A,P05C07A,P05C07B,P05C16,P05D01,P05H01A,OCUPADOS
0,2025,2,2,13796,338,3,14,3.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,2,2,13796,338,1,42,5.0,23.0,0.0,6.0,NaN,35.0,1.0
2,2025,2,2,13796,338,2,42,5.0,20.0,0.0,1.0,7000.0,1.0,1.0
3,2025,2,2,13796,338,5,9,2.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,2,2,13796,338,4,11,2.0,NaN,NaN,NaN,NaN,NaN,NaN
5,2025,2,2,5636,580,4,22,4.0,NaN,NaN,NaN,NaN,NaN,NaN
6,2025,2,2,5636,580,2,41,2.0,0.0,3.0,4.0,800.0,25.0,1.0
7,2025,2,2,5636,580,5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2025,2,2,5636,580,1,45,2.0,0.0,1.0,7.0,NaN,25.0,1.0
9,2025,2,2,5636,580,3,20,4.0,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
for columna in COLUMNAS_REQUERIDAS:
    print(f"\n--- {columna} ---")
    print("Tipo:", pdf_t1[columna].dtype)
    print("Ejemplos:", pdf_t1[columna].dropna().unique()[:10])


--- P05D01 ---
Tipo: float64
Ejemplos: [7000.  800. 2300. 3000. 1600. 3400. 2500. 3800.  600. 1920.]

--- P02A03 ---
Tipo: int64
Ejemplos: [14 42  9 11 22 41  0 45 20 31]

--- P05C07A ---
Tipo: float64
Ejemplos: [23. 20.  0.  5.  6. 52.  1. 35. 25.  3.]

--- P05C07B ---
Tipo: float64
Ejemplos: [ 0.  3.  1.  6.  2.  5.  9. 11. 10.  7.]

--- P05H01A ---
Tipo: float64
Ejemplos: [35.  1. 25. 21. 36. 20. 24. 63. 42. 77.]

--- P03A03A ---
Tipo: float64
Ejemplos: [3. 5. 2. 4. 0. 1. 6. 7.]

--- P05C16 ---
Tipo: float64
Ejemplos: [6. 1. 4. 7. 5. 3. 2. 9. 8.]

--- DOMINIO ---
Tipo: int64
Ejemplos: [2 3 1]

--- OCUPADOS ---
Tipo: float64
Ejemplos: [1.]

--- NUM_HOGAR ---
Tipo: int64
Ejemplos: [13796  5636  8825 13797 13798  2713  9465  5052  8784 13799]

--- NUM_PERSONA ---
Tipo: int64
Ejemplos: [ 3  1  2  5  4  7  6  9  8 10]

--- FACTOR ---
Tipo: int64
Ejemplos: [338 580 562 216 552 181  68 620 200 686]

--- ANIO ---
Tipo: int64
Ejemplos: [2025]

--- TRIMESTRE ---
Tipo: int64
Ejemplos: [2]


In [19]:
print(pdf_t1["ANIO"].value_counts(dropna=False))

print(pdf_t1["TRIMESTRE"].value_counts(dropna=False))

ANIO
2025    51588
Name: count, dtype: int64
TRIMESTRE
2    51588
Name: count, dtype: int64


In [20]:
COLUMNAS_NUMERICAS = [
    "P05D01",
    "P02A03",
    "P05C07A",
    "P05C07B",
    "P05H01A",
    "OCUPADOS",
    "FACTOR",
    "ANIO",
    "TRIMESTRE"
]

COLUMNAS_CATEGORICAS = [
    "P03A03A",
    "P05C16",
    "DOMINIO",
    "NUM_HOGAR",
    "NUM_PERSONA"
]

In [24]:
def cargar_y_armonizar_excel(periodo, info):

    print(f"Procesando {periodo}...")

    # Leer solamente las columnas necesarias
    pdf = pd.read_excel(
        info["ruta"],
        engine="openpyxl",
        usecols=COLUMNAS_REQUERIDAS
    )

    registros_originales = len(pdf)

    # Homologar variables numéricas
    for col in COLUMNAS_NUMERICAS:
        pdf[col] = pd.to_numeric(
            pdf[col],
            errors="coerce"
        )

    # Homologar códigos categóricos
    for col in COLUMNAS_CATEGORICAS:

        pdf[col] = pdf[col].astype("string")

        # Ejemplo: "1.0" -> "1"
        pdf[col] = pdf[col].str.replace(
            r"\.0$",
            "",
            regex=True
        )

    # Renombrar variables analíticas
    pdf = pdf.rename(columns=RENOMBRES)

    # Variables de trazabilidad
    pdf["archivo_origen"] = os.path.basename(info["ruta"])
    pdf["periodo_archivo"] = periodo
    pdf["anio_archivo"] = int(info["anio"])
    pdf["trimestre_calendario"] = int(info["trimestre"])

    print(
        f"{periodo}: "
        f"{registros_originales:,} registros"
    )

    return pdf

In [22]:
prueba_t1 = cargar_y_armonizar_excel(
    "2025T1",
    ARCHIVOS["2025T1"]
)

print(prueba_t1.shape)

display(prueba_t1.head())

Procesando 2025T1...
2025T1: 51,588 registros
(51588, 18)


,ANIO,TRIMESTRE,dominio,NUM_HOGAR,FACTOR,NUM_PERSONA,edad,nivel_educativo,antiguedad_anios,antiguedad_meses,categoria_ocupacional,salario_mensual,horas_semanales,ocupado,archivo_origen,periodo_archivo,anio_archivo,trimestre_calendario
0,2025,2,2,13796,338,3,14,3,NaN,NaN,<NA>,NaN,NaN,NaN,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1
1,2025,2,2,13796,338,1,42,5,23.0,0.0,6,NaN,35.0,1.0,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1
2,2025,2,2,13796,338,2,42,5,20.0,0.0,1,7000.0,1.0,1.0,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1
3,2025,2,2,13796,338,5,9,2,NaN,NaN,<NA>,NaN,NaN,NaN,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1
4,2025,2,2,13796,338,4,11,2,NaN,NaN,<NA>,NaN,NaN,NaN,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1


In [23]:
display(
    prueba_t1[
        [
            "ANIO",
            "TRIMESTRE",
            "archivo_origen",
            "periodo_archivo",
            "anio_archivo",
            "trimestre_calendario"
        ]
    ].head()
)

,ANIO,TRIMESTRE,archivo_origen,periodo_archivo,anio_archivo,trimestre_calendario
0,2025,2,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1
1,2025,2,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1
2,2025,2,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1
3,2025,2,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1
4,2025,2,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1


In [26]:
conteos_originales = {}

for periodo, info in ARCHIVOS.items():

    print("\n" + "=" * 50)
    print(f"INICIANDO {periodo}")
    print("=" * 50)

    # 1. Excel -> pandas
    pdf = cargar_y_armonizar_excel(periodo, info)

    conteos_originales[periodo] = len(pdf)

    # 2. Cambiar pd.NA / NaN por None donde corresponda
    pdf_spark = pdf.astype(object).where(pd.notnull(pdf), None)

    # 3. pandas -> Spark
    sdf = spark.createDataFrame(pdf_spark)

    # 4. Guardar como Parquet
    ruta_salida = f"{PARQUET_DIR}/{periodo}"

    (
        sdf.write
        .mode("overwrite")
        .parquet(ruta_salida)
    )

    print(f"Parquet guardado en: {ruta_salida}")
    print(f"Registros: {sdf.count():,}")

    # 5. Liberar memoria
    del pdf
    del pdf_spark
    del sdf

    gc.collect()

print("\nConversión terminada.")


INICIANDO 2025T1
Procesando 2025T1...
2025T1: 51,588 registros


26/09/24 22:31:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/24 22:31:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/24 22:31:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/24 22:31:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/09/24 22:31:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/09/24 22:31:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/09/24 22:31:53 WARN MemoryManager: Total allocation exceeds 95.

Parquet guardado en: ./data/parquet/2025T1
Registros: 51,588

INICIANDO 2025T2
Procesando 2025T2...
2025T2: 51,167 registros


26/09/24 22:32:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/24 22:32:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/24 22:32:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/24 22:32:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/24 22:32:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/24 22:32:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Parquet guardado en: ./data/parquet/2025T2
Registros: 51,167

INICIANDO 2025T3
Procesando 2025T3...
2025T3: 51,583 registros


...

Parquet guardado en: ./data/parquet/2025T3
Registros: 51,583

INICIANDO 2025T4
Procesando 2025T4...
2025T4: 49,338 registros


26/09/24 22:34:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/24 22:34:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/24 22:34:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/24 22:34:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/09/24 22:34:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/09/24 22:34:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/09/24 22:34:07 WARN MemoryManager: Total allocation exceeds 95.

Parquet guardado en: ./data/parquet/2025T4
Registros: 49,338

INICIANDO 2026T1
Procesando 2026T1...
2026T1: 49,843 registros


26/09/24 22:34:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/24 22:34:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/24 22:34:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/24 22:34:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/09/24 22:34:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/09/24 22:34:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/09/24 22:34:58 WARN MemoryManager: Total allocation exceeds 95.

Parquet guardado en: ./data/parquet/2026T1
Registros: 49,843

Conversión terminada.


In [27]:
print("REGISTROS ORIGINALES")
print("-" * 35)

for periodo, cantidad in conteos_originales.items():
    print(f"{periodo}: {cantidad:,}")

REGISTROS ORIGINALES
-----------------------------------
2025T1: 51,588
2025T2: 51,167
2025T3: 51,583
2025T4: 49,338
2026T1: 49,843


In [28]:
df_2025_t1 = spark.read.parquet(f"{PARQUET_DIR}/2025T1")
df_2025_t2 = spark.read.parquet(f"{PARQUET_DIR}/2025T2")
df_2025_t3 = spark.read.parquet(f"{PARQUET_DIR}/2025T3")
df_2025_t4 = spark.read.parquet(f"{PARQUET_DIR}/2025T4")

df_2026 = spark.read.parquet(f"{PARQUET_DIR}/2026T1")

In [29]:
df_2025 = (
    df_2025_t1
    .unionByName(df_2025_t2)
    .unionByName(df_2025_t3)
    .unionByName(df_2025_t4)
)

In [30]:
total_2025 = df_2025.count()

print(f"Total registros 2025: {total_2025:,}")

Total registros 2025: 203,676


In [31]:
df_2025.printSchema()

root
 |-- ANIO: long (nullable = true)
 |-- TRIMESTRE: long (nullable = true)
 |-- dominio: string (nullable = true)
 |-- NUM_HOGAR: string (nullable = true)
 |-- FACTOR: long (nullable = true)
 |-- NUM_PERSONA: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- nivel_educativo: string (nullable = true)
 |-- antiguedad_anios: double (nullable = true)
 |-- antiguedad_meses: double (nullable = true)
 |-- categoria_ocupacional: string (nullable = true)
 |-- salario_mensual: double (nullable = true)
 |-- horas_semanales: double (nullable = true)
 |-- ocupado: double (nullable = true)
 |-- archivo_origen: string (nullable = true)
 |-- periodo_archivo: string (nullable = true)
 |-- anio_archivo: long (nullable = true)
 |-- trimestre_calendario: long (nullable = true)



In [32]:
df_2025.show(5, truncate=False)

+----+---------+-------+---------+------+-----------+----+---------------+----------------+----------------+---------------------+---------------+---------------+-------+---------------------------+---------------+------------+--------------------+
|ANIO|TRIMESTRE|dominio|NUM_HOGAR|FACTOR|NUM_PERSONA|edad|nivel_educativo|antiguedad_anios|antiguedad_meses|categoria_ocupacional|salario_mensual|horas_semanales|ocupado|archivo_origen             |periodo_archivo|anio_archivo|trimestre_calendario|
+----+---------+-------+---------+------+-----------+----+---------------+----------------+----------------+---------------------+---------------+---------------+-------+---------------------------+---------------+------------+--------------------+
|2025|2        |2      |16033    |356   |2          |44  |5              |6.0             |0.0             |1                    |5691.0         |50.0           |1.0    |Personas_ENEIC_T1_2025.xlsx|2025T1         |2025        |1                   |
|202

In [33]:
(
    df_2025
    .groupBy(
        "periodo_archivo",
        "anio_archivo",
        "trimestre_calendario"
    )
    .count()
    .orderBy("periodo_archivo")
    .show()
)

+---------------+------------+--------------------+-----+
|periodo_archivo|anio_archivo|trimestre_calendario|count|
+---------------+------------+--------------------+-----+
|         2025T1|        2025|                   1|51588|
|         2025T2|        2025|                   2|51167|
|         2025T3|        2025|                   3|51583|
|         2025T4|        2025|                   4|49338|
+---------------+------------+--------------------+-----+



In [34]:
(
    df_2025
    .groupBy(
        "periodo_archivo",
        "TRIMESTRE"
    )
    .count()
    .orderBy(
        "periodo_archivo",
        "TRIMESTRE"
    )
    .show()
)

+---------------+---------+-----+
|periodo_archivo|TRIMESTRE|count|
+---------------+---------+-----+
|         2025T1|        2|51588|
|         2025T2|        2|  175|
|         2025T2|        3|50992|
|         2025T3|        4|51583|
|         2025T4|        5|49338|
+---------------+---------+-----+



### Identificación del período

Se conservó la variable `TRIMESTRE` tal como aparece en las bases
originales. Sin embargo, esta variable no se utilizó directamente para
identificar el trimestre calendario.

En su lugar, se construyeron las variables `periodo_archivo`,
`anio_archivo` y `trimestre_calendario` utilizando el archivo de
procedencia de cada observación.

Esta decisión evita asumir que los códigos originales de `TRIMESTRE`
corresponden directamente al trimestre calendario y permite mantener
trazabilidad respecto de la fuente original.

In [35]:
columnas_analisis = [
    "salario_mensual",
    "edad",
    "antiguedad_anios",
    "antiguedad_meses",
    "horas_semanales",
    "nivel_educativo",
    "categoria_ocupacional",
    "dominio",
    "ocupado",
    "NUM_HOGAR",
    "NUM_PERSONA",
    "FACTOR",
    "ANIO",
    "TRIMESTRE"
]

total = df_2025.count()

faltantes_expr = []

for col in columnas_analisis:
    faltantes_expr.append(
        F.sum(
            F.when(F.col(col).isNull(), 1).otherwise(0)
        ).alias(col)
    )

fila_faltantes = df_2025.agg(*faltantes_expr).first()

datos_faltantes = []

for col in columnas_analisis:
    n = fila_faltantes[col]

    datos_faltantes.append(
        (
            col,
            n,
            round(n / total * 100, 2)
        )
    )

df_faltantes = spark.createDataFrame(
    datos_faltantes,
    ["variable", "faltantes", "porcentaje"]
)

df_faltantes.show(30, truncate=False)

+---------------------+---------+----------+
|variable             |faltantes|porcentaje|
+---------------------+---------+----------+
|salario_mensual      |150651   |73.97     |
|edad                 |0        |0.0       |
|antiguedad_anios     |115454   |56.69     |
|antiguedad_meses     |115454   |56.69     |
|horas_semanales      |115454   |56.69     |
|nivel_educativo      |26344    |12.93     |
|categoria_ocupacional|115454   |56.69     |
|dominio              |0        |0.0       |
|ocupado              |115454   |56.69     |
|NUM_HOGAR            |0        |0.0       |
|NUM_PERSONA          |0        |0.0       |
|FACTOR               |0        |0.0       |
|ANIO                 |0        |0.0       |
|TRIMESTRE            |0        |0.0       |
+---------------------+---------+----------+



In [36]:
claves = [
    "periodo_archivo",
    "NUM_HOGAR",
    "NUM_PERSONA"
]

duplicados = (
    df_2025
    .groupBy(*claves)
    .count()
    .filter(F.col("count") > 1)
)

cantidad_claves_duplicadas = duplicados.count()

print(
    "Cantidad de claves con más de un registro:",
    cantidad_claves_duplicadas
)

[Stage 44:======================================>                (45 + 19) / 64]

Cantidad de claves con más de un registro: 0


In [37]:
duplicados.orderBy(F.desc("count")).show(
    20,
    truncate=False
)

[Stage 50:===============================>                       (37 + 16) / 64]

+---------------+---------+-----------+-----+
|periodo_archivo|NUM_HOGAR|NUM_PERSONA|count|
+---------------+---------+-----------+-----+
+---------------+---------+-----------+-----+

